[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter03_from_neurons_to_a_network/notebook.ipynb)

# Chapter 3: From Neurons to a Network

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Frank Rosenblatt and Marvin Minsky had known each other since they were teenagers, because Minsky was a year behind Rosenblatt at the Bronx High School of Science in New York. Both grew up to study intelligence, and both believed machines could one day think, but they disagreed completely about how to get there. Rosenblatt believed in machines that learned their own behaviour from examples, the way his Perceptron did. Minsky, who helped to found the artificial intelligence laboratory at the Massachusetts Institute of Technology, believed that intelligence would come from carefully written rules and symbols, and he had grown tired of what he saw as wild promises being made for the Perceptron.

In 1969, Minsky and his MIT colleague Seymour Papert published a book called *Perceptrons*. It was careful, mathematical and, in places, admiring, but its most famous result was a list of simple problems that a single layer of Perceptron neurons could never solve, no matter how long it trained. The simplest of them is a puzzle so small that you can write all of it on the back of a bus ticket. It has just two inputs, each of which is either 0 or 1, and it asks for an answer of 1 only when the two inputs are different. That puzzle is called **XOR**.

Minsky and Papert knew that a network with more than one layer could solve XOR. What nobody knew in 1969 was how such a network could be trained, and the book doubted that anyone would ever find out. Its message, as it spread, became something much simpler and much harsher, which was that neural networks did not work. Funding for neural network research dried up over the following years, in what came to be called the first AI winter. Rosenblatt never saw the thaw, because in 1971 he drowned in a sailing accident on Chesapeake Bay, on his 43rd birthday.

The answer to Minsky and Papert's puzzle turned out to be exactly what they suspected, which was to stop using one neuron and start building networks. This chapter builds one. You will stack neurons into layers, stack layers into a network, run a small network forward with nothing but a pencil, and then see for yourself why the XOR puzzle defeats a single neuron but not a network of three. Before you read on, try to guess the answers to these four questions.

1. **If several neurons all read the same inputs, how could they still give different answers?**
2. **What does the word "hidden" mean in a hidden layer?**
3. **How much arithmetic does it take to run a small network once, from its inputs to its answer?**
4. **Why can one neuron not solve XOR, when it only has four possible inputs?**

## Stacking Neurons into a Layer

In Chapter 2 a single neuron decided whether a student was ready for an exam. A real school would rarely trust one opinion, so imagine three teachers judging the same student, each with a different view of what matters. The first teacher cares mostly about study, the second cares mostly about sleep, and the third weighs both equally but is harder to impress. Each teacher is one neuron, and each has their own weights and their own bias.

- **Teacher 1** gives study a weight of 0.9 and sleep a weight of 0.1, with a bias of $-4$.
- **Teacher 2** gives study a weight of 0.1 and sleep a weight of 0.9, with a bias of $-4$.
- **Teacher 3** gives both a weight of 0.5, with a stricter bias of $-5$.

A group of neurons that all read the same inputs, side by side, is called a **layer**. The neat trick is that a whole layer's weights fit into a single table of numbers called a **weight matrix**, with one row for each neuron and one column for each input. Figure 3.1 shows the three teachers as a layer and, next to them, the weight matrix that holds all six of their weights.

> 🖼️ **Figure 3.1:** Three teachers form a layer of three neurons. All six of their weights fit into one weight matrix, where each coloured row belongs to one neuron.

Holding the weights as a matrix pays off immediately in code, because numpy can multiply a matrix by a list of inputs in one step. The first cell sets up a student who studied for 8 hours but slept only 3 hours a night, along with the matrix and the three biases.

In [ ]:
import numpy as np

x = np.array([8, 3])            # hours studied, hours slept
W = np.array([[0.9, 0.1],
              [0.1, 0.9],
              [0.5, 0.5]])
b = np.array([-4.0, -4.0, -5.0])

The expression `W @ x` computes the weighted sum for every row of the matrix at once, which means it runs all three neurons together. Adding the biases and checking which sums are above zero gives each teacher's verdict.

In [ ]:
z = W @ x + b
print(z)
print((z > 0).astype(int))

The teachers disagree, and that is the whole point of a layer. The teacher who values study says yes, the teacher who values sleep says no, and the strict balanced teacher only just says yes. **A layer is several neurons reading the same inputs, each with its own weights, so that together they describe the input from several angles at once.** The sizes of the pieces always follow the same rule, and checking them is a habit worth building right now.

In [ ]:
print(W.shape, x.shape, z.shape)

A matrix with 3 rows and 2 columns takes in 2 numbers and gives back 3. The number of columns must match the number of inputs, and the number of rows sets how many outputs the layer produces, one for each of its neurons.

## Hidden Layers and Depth

The three verdicts that came out of the layer are just three new numbers, and nothing stops you from handing them to another layer as its inputs. That is how a network is built. Think of a busy restaurant kitchen, where one cook washes and chops the vegetables, a second cook turns them into a sauce, and a third puts everything on the plate. No cook does the whole dish, and each one works only with what the cook before them handed over.

The first layer reads the raw inputs, and the last layer, called the **output layer**, gives the network's final answer. Every layer in between is a **hidden layer**, because you never set or see its numbers directly, since it only talks to the layers on either side of it. The number of neurons in a layer is called its **width**, and the number of layers is the network's **depth**. Figure 3.2 shows a network with two inputs, two hidden layers of three neurons each, and one output.

> 🖼️ **Figure 3.2:** A network of depth three. Each layer has its own weight matrix, and the shape of each matrix is set by the widths of the two layers it joins.

Each pair of neighbouring layers is joined by its own weight matrix, and the shape rule from the last section tells you its size. A matrix that joins a layer of 2 to a layer of 3 has 3 rows and 2 columns, so it holds 6 weights, and the second layer also needs 3 biases. Table 3.1 goes through the whole network in this way.

**Table 3.1: Every weight and bias in the network of Figure 3.2, counted one connection at a time.**

| Connection | Matrix shape | Weights | Biases | Total |
|---|---|---|---|---|
| Input to hidden 1 | $3\times2$ | 6 | 3 | 9 |
| Hidden 1 to hidden 2 | $3\times3$ | 9 | 3 | 12 |
| Hidden 2 to output | $1\times3$ | 3 | 1 | 4 |
| Whole network |  | 18 | 7 | 25 |

Counting by hand works for a small network, but a few lines of code can count any network from a list of its layer widths. Each connection contributes one weight for every pair of neurons it joins and one bias for every neuron it feeds.

In [ ]:
sizes = [2, 3, 3, 1]
total = 0
for n_in, n_out in zip(sizes[:-1], sizes[1:]):
    count = n_out * n_in + n_out
    print(n_in, "to", n_out, ":", count)
    total += count
print(total)

Those 25 adjustable numbers are everything this network knows. If you give the same code the sizes `[2, 8, 1]` of the network from Chapter 1, it prints 33, the same count that scikit learn reported for the plants in the field. **A network is a chain of layers in which each layer's outputs become the next layer's inputs, and every connection between two layers is one weight matrix.**

## The Forward Pass by Hand

Running a network once, from its inputs all the way to its answer, is called the **forward pass**. It needs nothing more than what you already know, which is to compute one layer, hand its outputs to the next layer, and repeat until you reach the end. The best way to understand it is to do it once entirely by hand.

Figure 3.3 shows a tiny network with two inputs, a hidden layer of two neurons and one output neuron, all using the step activation from Chapter 2. Every weight is written on its line and every bias is written under its neuron. Do not worry yet about what this network is for, because working that out is part of the fun. The figure follows the input pair 1 and 0.

> 🖼️ **Figure 3.3:** A forward pass by hand for the inputs 1 and 0. Each neuron adds its weighted inputs and its bias, and the step activation turns any total above zero into a 1.

Follow the numbers in the figure from left to right. The first hidden neuron, $h_1$, receives 1 times 1 from the first input and 1 times 0 from the second, which adds up to 1, and its bias of $-0.5$ brings the total to 0.5. That is above zero, so $h_1$ outputs 1. The second hidden neuron, $h_2$, has weights of $-1$, so it receives $-1$ and 0, and its bias of $1.5$ brings its total to 0.5 as well, so it also outputs 1. Finally the output neuron receives two 1s, each with a weight of 1, and its bias of $-1.5$ leaves a total of 0.5, so the network's answer is 1.

That was the whole forward pass, and it took only a few additions. In code, the same network is two weight matrices, two sets of biases and a step function that now works on a whole list of numbers at once.

In [ ]:
def step(z):
    return (z > 0).astype(int)

W1 = np.array([[ 1.0,  1.0],
               [-1.0, -1.0]])
b1 = np.array([-0.5, 1.5])
W2 = np.array([1.0, 1.0])
b2 = -1.5

The forward pass itself is just two lines, one for each layer, and the hidden layer's output `h` is passed straight into the output layer.

In [ ]:
def forward(x):
    h = step(W1 @ x + b1)
    y = step(W2 @ h + b2)
    return h, y

Now run all four possible pairs of inputs through the network and look for the pattern.

In [ ]:
for pair in ([0, 0], [0, 1], [1, 0], [1, 1]):
    h, y = forward(np.array(pair))
    print(pair, h, y)

The network answers 1 when exactly one of its inputs is 1, and 0 when the inputs match. That is XOR, the very puzzle from the opening of this chapter. Table 3.2 lays the four forward passes side by side and also shows what each hidden neuron has learned to notice.

**Table 3.2: The forward pass for all four inputs. The first hidden neuron asks "either", the second asks "not both", and the output neuron says yes only when both answers are yes.**

| Inputs | $h_1$: is either input 1? | $h_2$: are they not both 1? | Output: are both $h$ equal to 1? | XOR |
|---|---|---|---|---|
| 0 and 0 | 0 | 1 | 0 | 0 |
| 0 and 1 | 1 | 1 | 1 | 1 |
| 1 and 0 | 1 | 1 | 1 | 1 |
| 1 and 1 | 1 | 0 | 0 | 0 |

Read the table as a short conversation. The first hidden neuron answers the question "is at least one input switched on?", the second answers "are the inputs not both switched on?", and the output neuron says yes only when both hidden answers are yes. Put together, that means one or the other but not both. You use exactly this logic every day, if your home has a staircase light with a switch at the bottom and another at the top, because flipping either switch changes the light. **A forward pass is just one layer after another, each one turning the previous layer's numbers into new ones, until the last layer gives the answer.**

## Why One Layer Cannot Solve XOR

The network you just ran used three neurons, so a fair question is whether one cleverly tuned neuron could have done the job alone. Chapter 2 already holds the answer, because a single neuron always splits its inputs with one straight line.

The XOR puzzle has only four possible inputs, and you can draw them as the four corners of a square. The two corners where the inputs differ, 0 and 1 or 1 and 0, should give 1, and they sit on opposite corners. The two corners where the inputs match, 0 and 0 or 1 and 1, should give 0, and they sit on the other two opposite corners. Try any straight line you like, and you will find that it always gets at least one corner wrong.

A computer can try a great many lines for you. The next cell creates 100,000 random neurons and records the best score that any of them achieves on the four XOR inputs.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
target = np.array([0, 1, 1, 0])
rng = np.random.default_rng(0)

best = 0
for trial in range(100_000):
    w = rng.uniform(-5, 5, 2)
    bias = rng.uniform(-5, 5)
    guess = ((X @ w + bias) > 0).astype(int)
    best = max(best, (guess == target).mean())
print(best)

Not one of the 100,000 neurons managed better than three corners out of four. Trying random lines is not a proof, but Minsky and Papert proved it properly, and the reason is exactly what the picture suggests. No single straight line can separate two opposite corners of a square from the other two.

So how did the small network succeed? The answer is the most important idea in this chapter. The hidden layer does not solve the puzzle directly. Instead it moves the points to new places where the puzzle becomes easy. Each input pair is turned into a new pair of numbers, $h_1$ and $h_2$, and in that new space the two corners that should give 1 both land in the same spot, while the two that should give 0 land somewhere else. Figure 3.4 shows the four points before and after the hidden layer.

> 🖼️ **Figure 3.4:** The four XOR inputs before and after the hidden layer. On the left no straight line can separate the circles from the triangles. On the right both circles have moved to the same corner, and a single line separates them easily.

On the right of the figure, the output neuron only has to draw one straight line, which is something a single neuron can always do. The hidden layer did the hard part by rearranging the problem until a straight line was enough. **A hidden layer reshapes its inputs into new numbers in which a hard problem becomes an easy one, and that is the real reason networks can do what single neurons cannot.** This is also what the plants in Chapter 1 needed. No straight line could separate the circle from the ring, so the network's eight hidden neurons reshaped the field until one could.

Two puzzles remain, and they are exactly the ones that kept neural networks frozen in 1969. In this chapter you chose the weights for the XOR network yourself, and a real network with thousands of weights needs a way to find them automatically, which is the subject of Chapters 5, 6 and 7. The step function causes trouble as well, because, as Chapter 2 showed, a small change to a weight usually changes nothing in its output, and Chapter 4 replaces it with bends that let a network learn.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Ask a fourth teacher.** Add a fourth row to `W` and a fourth bias to `b` for a teacher who cares only about study, then run the layer again. Check that `z` now has 4 values and that `W.shape` has become `(4, 2)`.
2. **Count a bigger network.** Use the counting cell to find how many weights and biases a network with the widths `[10, 64, 32, 1]` has. Then add one more hidden layer of 32 and see how much the total grows.
3. **Build AND and OR from one neuron.** Using the `step` function, find weights and a bias for a single neuron that outputs 1 only when both inputs are 1, which is called AND, and then another that outputs 1 when at least one input is 1, called OR. Explain with a sketch of the square why one neuron is enough for each of these.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does each row and each column of a weight matrix stand for?**
2. **Why is a hidden layer called hidden?**
3. **What is a forward pass, in one sentence?**
4. **Why can no single neuron solve XOR?**
5. **What does the hidden layer in the XOR network actually do to the four inputs?**

### Answers

1. Each row holds the weights of one neuron in the layer, and each column belongs to one of the layer's inputs.
2. Because you never set or see its numbers directly, since it only takes in the previous layer's outputs and passes its own outputs on to the next layer.
3. It is running a network once from its inputs to its answer, one layer at a time.
4. Because a single neuron can only split its inputs with one straight line, and no straight line can separate two opposite corners of a square from the other two.
5. It moves them to new values of $h_1$ and $h_2$, where one straight line can separate them.

## What Comes Next

You can now build layers and networks, run a forward pass, and explain why hidden layers matter.

- **Chapter 4, Activation Functions and Nonlinearity,** explains why every neuron needs a bend, and which bend revived deep networks.
- **Chapter 5, Loss Functions,** teaches a network to measure how wrong it is, the first step to finding weights.

Chapter 4 begins with a surprise, which is that many layers without bends are no stronger than one neuron.